# Textract EDS Adapter

This notebook processes PDF documents using AWS Textract with a custom adapter. It:
1. Reads the zero-shot results CSV to identify documents with forms
2. Extracts the lowest page number from each document's form_pages
3. Sends individual pages to Textract using the custom adapter


In [1]:
import pandas as pd
import boto3
import os
from pathlib import Path
import json
from typing import List, Dict, Any
import PyPDF2
from io import BytesIO
import base64

## Configuration

In [ ]:
# Configuration
CLOBBER = False  # Set to True to overwrite existing results, False to skip already processed files

# Agency filtering toggle - set to None to process all agencies, or specify agency name(s) to filter
# Examples:
# FILTER_AGENCIES = None  # Process all agencies
# FILTER_AGENCIES = "Education"  # Process only Education agency
# FILTER_AGENCIES = ["Education", "Correction"]  # Process multiple agencies
FILTER_AGENCIES = None #"Correction"

# File paths
CSV_PATH = "../../preprocessing/zero_shot_results_full_corpus.csv"
CONTRACTS_DIR = "../../../data/raw/_contracts/"
OUTPUT_DIR = "../../../data/intermediate_products/eds_forms_textract/"

# AWS Textract configuration
REGION_NAME = "us-east-1"  # Update with your region
CUSTOM_ADAPTER_ID = "4403a7771cbe"  # Update with your custom adapter ID
ADAPTER_VERSION = "2"  # Update with your adapter version

# Feature types - only QUERIES works with custom adapters
ADAPTER_FEATURE_TYPES = ['QUERIES']

# Queries for the EDS form extraction
DEFAULT_QUERIES = [
    "EDS Number",
    "Date Prepared",
    "Which response options in the \"3. CONTRACTS & LEASES\" box are checked?",
    #"Account Number",
    "Total amount this action:",
    "New contract total",
    #"Revenue generated this action:",
    #"Revenue generated total contract:",
    "From (month/day, year)",
    "To (month, day, year)",
    "Which option in the \"13. Method of source selection:\" box is selected?",
    "Name of agency:",
    #"Name listed for item \"17. Name\" under AGENCY CONTACT INFORMATION?",
    "Vendor ID #",
    "Vendor Name",
    #"Is \"Yes\" checked for 'Is the vendor registered with the Secretary of State'?",
    "Is \"Yes\" checked for Primary Vendor: Minority:'?",
    "Is \"Yes\" checked for Primary Vendor: Women:'?",
    #"Is \"Yes\" checked for Sub Vendor: Minority:'?",
    #"Is \"Yes\" checked for Sub Vendor: Women:'?",
    "Is \"Yes\" checked for 'Is there Renewal Language in the document'?",
    "Is \"Yes\" checked for 'Is there a Termination for Convenience\" clause in the document'?"
]

# Initialize AWS client
textract_client = boto3.client('textract', region_name=REGION_NAME)

## Load and Filter Data

In [ ]:
# Load professional services contracts data
with open("../../../data/raw/indiana_prof_services_contracts.json", 'r') as f:
    prof_services_contracts = json.load(f)

print(f"Total professional services contracts: {len(prof_services_contracts)}")
[contract['agencyName'] for contract in prof_services_contracts].count('Correction')
#tabulate([contract['agencyName'] for contract in prof_services_contracts])

In [17]:


# Apply agency filter if specified
if FILTER_AGENCIES is not None:
    print(f"\n=== AGENCY FILTER APPLIED ===")
    
    # Handle both single agency string and list of agencies
    if isinstance(FILTER_AGENCIES, str):
        target_agencies = [FILTER_AGENCIES]
    else:
        target_agencies = FILTER_AGENCIES
    
    print(f"Filtering for agencies: {target_agencies}")
    
    # Filter contracts by agency
    original_count = len(prof_services_contracts)
    prof_services_contracts = [
        contract for contract in prof_services_contracts 
        if contract['agencyName'] in target_agencies
    ]
    
    print(f"Contracts after agency filter: {len(prof_services_contracts)} (reduced from {original_count})")
    
    # Show agency distribution
    agency_counts = {}
    for contract in prof_services_contracts:
        agency = contract['agencyName']
        agency_counts[agency] = agency_counts.get(agency, 0) + 1
    
    print("Agency distribution:")
    for agency, count in sorted(agency_counts.items()):
        print(f"  {agency}: {count} contracts")
    print("=" * 30)
else:
    print("No agency filter applied - processing all agencies")

# Extract PDF filenames from professional services contracts
prof_services_filenames = set()
for contract in prof_services_contracts:
    pdf_url = contract['pdfUrl']
    # Extract filename from URL (e.g., "0000000000000000000011571-011.pdf")
    filename = pdf_url.split('/')[-1]
    prof_services_filenames.add(filename)

print(f"Unique professional services contract PDF files: {len(prof_services_filenames)}")

# Load the CSV file
df = pd.read_csv(CSV_PATH)
print(f"Total documents in CSV: {len(df)}")

# Filter for documents containing forms AND are professional services contracts
forms_df = df[df['contains_form'] == True].copy()
print(f"Documents with forms (before prof services filter): {len(forms_df)}")

# Filter to only include professional services contracts
prof_services_forms_df = forms_df[forms_df['filename'].isin(prof_services_filenames)].copy()
print(f"Professional services documents with forms: {len(prof_services_forms_df)}")

# Display sample of filtered data
print("\nSample of professional services documents with forms:")
print(prof_services_forms_df[['filename', 'form_pages', 'num_form_pages']].head())

# Update the working dataframe for the rest of the notebook
forms_df = prof_services_forms_df

No agency filter applied - processing all agencies
Unique professional services contract PDF files: 35300
Total documents in CSV: 160751
Documents with forms (before prof services filter): 68696
Professional services documents with forms: 26108

Sample of professional services documents with forms:
                                filename form_pages  num_form_pages
3996   0000000000000000000031808-000.pdf          1               1
9423   0000000000000000000040451-000.pdf      16,35               2
10225  0000000000000000000041271-001.pdf         17               1
11065  0000000000000000000048512-002.pdf          1               1
12541  0000000000000000000055177-000.pdf         16               1


/var/folders/_0/grm8p2890sj6kq7p40wjd3pw0000gn/T/ipykernel_82282/2674984664.py:46: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


## Helper Functions

In [19]:
def get_lowest_page_number(form_pages_str: str) -> int:
    """
    Extract the lowest page number from the form_pages string.
    
    Args:
        form_pages_str: String containing page numbers (e.g., "1,2,4,10,15,16" or "5")
    
    Returns:
        int: The lowest page number
    """
    if pd.isna(form_pages_str) or form_pages_str == "":
        return None
    
    # Handle both single numbers and comma-separated lists
    if ',' in str(form_pages_str):
        page_numbers = [int(x.strip()) for x in str(form_pages_str).split(',')]
    else:
        page_numbers = [int(str(form_pages_str).strip())]
    
    return min(page_numbers)

def get_output_filename(filename: str, page_number: int) -> str:
    """
    Generate the output JSON filename for a given PDF and page number.
    
    Args:
        filename: Original PDF filename
        page_number: Page number
    
    Returns:
        str: Output JSON filename
    """
    filename_base = Path(filename).stem
    return f"{filename_base}_page_{page_number}_textract.json"

def file_already_processed(filename: str, page_number: int, output_dir: Path) -> bool:
    """
    Check if a file has already been processed.
    
    Args:
        filename: Original PDF filename
        page_number: Page number
        output_dir: Output directory path
    
    Returns:
        bool: True if file exists and has been processed
    """
    json_filename = get_output_filename(filename, page_number)
    json_path = output_dir / json_filename
    return json_path.exists() and json_path.stat().st_size > 0

def extract_single_page_pdf(pdf_path: str, page_number: int) -> bytes:
    """
    Extract a single page from a PDF file.
    
    Args:
        pdf_path: Path to the PDF file
        page_number: Page number to extract (1-indexed)
    
    Returns:
        bytes: PDF content of the single page
    """
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        pdf_writer = PyPDF2.PdfWriter()
        
        # PyPDF2 uses 0-based indexing, so subtract 1
        pdf_writer.add_page(pdf_reader.pages[page_number - 1])
        
        output_buffer = BytesIO()
        pdf_writer.write(output_buffer)
        return output_buffer.getvalue()

def process_with_textract(pdf_bytes: bytes, adapter_id: str, queries: List[str] = None) -> Dict[str, Any]:
    """
    Process a PDF page with Textract using a custom adapter.
    
    Args:
        pdf_bytes: PDF content as bytes
        adapter_id: Custom adapter ID
        queries: List of queries for the adapter (uses DEFAULT_QUERIES if not provided)
    
    Returns:
        dict: Textract response
    """
    try:
        # Use provided queries or default queries
        if queries is None:
            queries = DEFAULT_QUERIES
        
        # Prepare the request parameters
        request_params = {
            'Document': {'Bytes': pdf_bytes},
            'FeatureTypes': ADAPTER_FEATURE_TYPES,
            'QueriesConfig': {
                'Queries': [{'Text': query} for query in queries]
            },
            'AdaptersConfig': {
                'Adapters': [
                    {
                        'AdapterId': adapter_id,
                        'Version': ADAPTER_VERSION
                    }
                ]
            }
        }
        
        response = textract_client.analyze_document(**request_params)
        return response
    except Exception as e:
        print(f"Error processing with Textract: {str(e)}")
        return None

In [23]:
# Add lowest page number to dataframe
forms_df['lowest_page'] = forms_df['form_pages'].apply(get_lowest_page_number)

# Remove rows where we couldn't determine the lowest page
forms_df = forms_df.dropna(subset=['lowest_page'])
forms_df['lowest_page'] = forms_df['lowest_page'].astype(int)

print(f"Documents with valid page numbers: {len(forms_df)}")
print("\nSample with lowest page numbers:")
print(forms_df[['filename', 'form_pages', 'lowest_page']].head(10))

# Check how many files remain unprocessed before creating output directory
print(f"\n=== PROCESSING QUEUE SUMMARY ===")

# Show agency filter status
if FILTER_AGENCIES is not None:
    if isinstance(FILTER_AGENCIES, str):
        print(f"🔍 AGENCY FILTER: {FILTER_AGENCIES}")
    else:
        print(f"🔍 AGENCY FILTER: {', '.join(FILTER_AGENCIES)}")
else:
    print("🔍 AGENCY FILTER: None (processing all agencies)")

print(f"Total professional services documents with forms queued: {len(forms_df)}")

# Create output directory if it doesn't exist to check existing files
temp_output_dir = Path(OUTPUT_DIR)
temp_output_dir.mkdir(parents=True, exist_ok=True)

# Count already processed files
already_processed_count = 0
for idx, row in forms_df.iterrows():
    filename = row['filename']
    lowest_page = row['lowest_page']
    if file_already_processed(filename, lowest_page, temp_output_dir):
        already_processed_count += 1

remaining_to_process = len(forms_df) - already_processed_count
print(f"Already processed: {already_processed_count}")
print(f"Remaining to process: {remaining_to_process}")
print(f"Processing progress: {(already_processed_count/len(forms_df)*100):.1f}% complete")
print("=" * 35)

Documents with valid page numbers: 26108

Sample with lowest page numbers:
                                filename                  form_pages  \
3996   0000000000000000000031808-000.pdf                           1   
9423   0000000000000000000040451-000.pdf                       16,35   
10225  0000000000000000000041271-001.pdf                          17   
11065  0000000000000000000048512-002.pdf                           1   
12541  0000000000000000000055177-000.pdf                          16   
12999  0000000000000000000041742-000.pdf  156,670,671,1176,1690,1691   
13200  0000000000000000000067712-000.pdf                          16   
14923  0000000000000000000081653-000.pdf                          16   
15147  0000000000000000000082482-000.pdf                          16   
15351  0000000000000000000083053-000.pdf                       21,27   

       lowest_page  
3996             1  
9423            16  
10225           17  
11065            1  
12541           16  
12999 

In [25]:
# Add lowest page number to dataframe
forms_df['lowest_page'] = forms_df['form_pages'].apply(get_lowest_page_number)

# Remove rows where we couldn't determine the lowest page
forms_df = forms_df.dropna(subset=['lowest_page'])
forms_df['lowest_page'] = forms_df['lowest_page'].astype(int)

print(f"Documents with valid page numbers: {len(forms_df)}")
print("\nSample with lowest page numbers:")
print(forms_df[['filename', 'form_pages', 'lowest_page']].head(10))

Documents with valid page numbers: 26108

Sample with lowest page numbers:
                                filename                  form_pages  \
3996   0000000000000000000031808-000.pdf                           1   
9423   0000000000000000000040451-000.pdf                       16,35   
10225  0000000000000000000041271-001.pdf                          17   
11065  0000000000000000000048512-002.pdf                           1   
12541  0000000000000000000055177-000.pdf                          16   
12999  0000000000000000000041742-000.pdf  156,670,671,1176,1690,1691   
13200  0000000000000000000067712-000.pdf                          16   
14923  0000000000000000000081653-000.pdf                          16   
15147  0000000000000000000082482-000.pdf                          16   
15351  0000000000000000000083053-000.pdf                       21,27   

       lowest_page  
3996             1  
9423            16  
10225           17  
11065            1  
12541           16  
12999 

## Save Results

In [29]:
# Create output directory if it doesn't exist
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Use the already processed count from the previous cell to filter dataframe
if not CLOBBER and already_processed_count > 0:
    print(f"CLOBBER = False: Filtering out {already_processed_count} already processed files...")
    # Filter out already processed files
    remaining_df = []
    for idx, row in forms_df.iterrows():
        filename = row['filename']
        lowest_page = row['lowest_page']
        if not file_already_processed(filename, lowest_page, output_dir):
            remaining_df.append(row)
    
    if remaining_df:
        forms_df = pd.DataFrame(remaining_df)
        print(f"Remaining files to process: {len(forms_df)}")
    else:
        print("All files already processed.")
        forms_df = pd.DataFrame()  # Empty dataframe
elif not CLOBBER:
    print("CLOBBER = False: No existing results found. Processing all files.")
else:
    print("CLOBBER = True: Processing all files (will overwrite existing results)")

# Process each document
results = []
errors = []
skipped = []

for idx, row in forms_df.iterrows():
    filename = row['filename']
    lowest_page = row['lowest_page']
    
    pdf_path = os.path.join(CONTRACTS_DIR, filename)
    
    # Check if file exists
    if not os.path.exists(pdf_path):
        error_msg = f"File not found: {filename}"
        print(error_msg)
        errors.append({'filename': filename, 'error': error_msg})
        continue
    
    try:
        print(f"Processing {filename}, page {lowest_page}...")
        
        # Extract the specific page
        page_pdf_bytes = extract_single_page_pdf(pdf_path, lowest_page)
        
        # Process with Textract
        textract_response = process_with_textract(page_pdf_bytes, CUSTOM_ADAPTER_ID)
        
        if textract_response:
            # Save result immediately to avoid losing work
            json_filename = get_output_filename(filename, lowest_page)
            json_path = output_dir / json_filename
            
            with open(json_path, 'w') as f:
                json.dump(textract_response, f, indent=2, default=str)
            
            result = {
                'filename': filename,
                'page_number': lowest_page,
                'json_file': json_filename,
                'status': 'success'
            }
            results.append(result)
            print(f"✓ Successfully processed {filename} → {json_filename}")
        else:
            error_msg = f"Textract processing failed for {filename}"
            print(f"✗ {error_msg}")
            errors.append({'filename': filename, 'error': error_msg})
            
    except Exception as e:
        error_msg = f"Error processing {filename}: {str(e)}"
        print(f"✗ {error_msg}")
        errors.append({'filename': filename, 'error': error_msg})

print(f"\n=== PROCESSING COMPLETE ===")
print(f"Successfully processed: {len(results)} documents")
print(f"Errors: {len(errors)} documents")
if len(results) + len(errors) > 0:
    success_rate = len(results)/(len(results)+len(errors))*100
    print(f"Success rate: {success_rate:.1f}%")

CLOBBER = False: Filtering out 2 already processed files...
Remaining files to process: 14
Processing 48262-010.pdf, page 1...
✓ Successfully processed 48262-010.pdf → 48262-010_page_1_textract.json
Processing 48262-012.pdf, page 1...
✓ Successfully processed 48262-012.pdf → 48262-012_page_1_textract.json
Processing 48262-011.pdf, page 1...
✓ Successfully processed 48262-011.pdf → 48262-011_page_1_textract.json
Processing 48262-001.pdf, page 1...
✓ Successfully processed 48262-001.pdf → 48262-001_page_1_textract.json
Processing 48262-008.pdf, page 1...
✓ Successfully processed 48262-008.pdf → 48262-008_page_1_textract.json
Processing 48262-009.pdf, page 1...
✓ Successfully processed 48262-009.pdf → 48262-009_page_1_textract.json
Processing 48262-007.pdf, page 1...
✓ Successfully processed 48262-007.pdf → 48262-007_page_1_textract.json
Processing 48262-005.pdf, page 1...
✓ Successfully processed 48262-005.pdf → 48262-005_page_1_textract.json
Processing 48262-004.pdf, page 1...
✓ Success

## Display Sample Results

In [112]:
# Save summary and error files
if results or already_processed_count > 0:
    # Count existing files if CLOBBER was False
    existing_files = []
    if not CLOBBER:
        for json_file in output_dir.glob("*.json"):
            if json_file.name != "processing_summary.json":
                existing_files.append(json_file.name)
    
    # Save summary of all results (including existing ones)
    summary = {
        'clobber_mode': CLOBBER,
        'agency_filter': FILTER_AGENCIES,
        'newly_processed_count': len(results),
        'error_count': len(errors),
        'total_existing_files': len(existing_files) if not CLOBBER else 0,
        'newly_processed_files': [{
            'filename': r['filename'],
            'page_number': r['page_number'],
            'json_file': r['json_file']
        } for r in results]
    }
    
    with open(output_dir / 'processing_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"Summary saved: {output_dir / 'processing_summary.json'}")

# Save errors if any
if errors:
    errors_df = pd.DataFrame(errors)
    errors_df.to_csv(output_dir / 'processing_errors.csv', index=False)
    print(f"Errors saved: {output_dir / 'processing_errors.csv'}")

print(f"Results saved to: {output_dir}")

Summary saved: ../../data/intermediate_products/eds_forms_textract/processing_summary.json
Results saved to: ../../data/intermediate_products/eds_forms_textract


## Error Analysis

In [ ]:
# Display sample extracted content from first successful result
if len(results) > 0:
    sample_result = results[0]
    print(f"\n=== SAMPLE EXTRACTION ===")
    print(f"File: {sample_result['filename']} (page {sample_result['page_number']})")
    
    # Load the JSON file to get Textract response
    json_path = output_dir / sample_result['json_file']
    with open(json_path, 'r') as f:
        textract_response = json.load(f)
    
    # Extract text blocks from Textract response
    textract_blocks = textract_response.get('Blocks', [])
    text_blocks = [block['Text'] for block in textract_blocks if block['BlockType'] == 'LINE']
    
    print("First 10 lines of detected text:")
    for i, text in enumerate(text_blocks[:10]):
        print(f"{i+1:2d}: {text}")
        
    # Show key-value pairs if detected
    key_value_blocks = [block for block in textract_blocks if block['BlockType'] == 'KEY_VALUE_SET']
    if key_value_blocks:
        print(f"\nDetected {len(key_value_blocks)} key-value pairs in the form.")
else:
    print("\n=== NO SAMPLE AVAILABLE ===")
    print("No files were successfully processed.")